# NB09 Guardian API Methodology (unstructured signal)

**Workstream:** Vishal + Viktor unstructured data / sentiment analysis

**API** the Guardian Open Platform API

**What** this notebook is for is

The notebook will link *structured* Companies House data with *unstructured* news data. We can't pull news for all 1.37M companies (the APIs simply won't allow that many calls), so instead we build **one reliable tool**: you give it a company, and it returns that company's news articles, a coverage count, and a sentiment score.

So This notebook **builds and tests that tool using the Guardian API**, on Viktor's ready-made 96-company sample. Once it works here, the exact same tool probably can be pointed at a targeted ~100-company sample (selected on structured signals like debt ratio) to produce a real result.

> Note nothing is final here until we have decided to move forward

Flow diagram to have a clear understanding

1.37M companies (structured Companies House data)
        │   filter on structured metrics (debt ratio, age, segment, etc.)
        ▼
   small set of companies  (~96 for testing (viktori) / ~100 for the real run)
        │
        ▼   ◄────────  THIS NOTEBOOK (Guardian source)
   for each company: we find news → score it → one tidy row
        │
        ▼   Then join back on CompanyNumber
   structured + unstructured  =  the combined, more powerful result
This notebook does the middle box, for the Guardian. For every company it runs the same small procedure:

search_name + town + sector
        │  build a precise Guardian query
        ▼
   call /search  ──►  response.total = how many articles exist (coverage count)
        │
        ▼  keep only articles that are genuinely about THIS company
   verify each hit (name in the body + a matching town / sector word)
        │
        ▼  run sentiment on the text we trust
   sentiment score + label
        │
        ▼
   one row:  CompanyNumber · articles · coverage_count · sentiment · …


_**Tool output**_ (experimental)

| Column | Meaning | Locaton |
|---|---|---|
| `CompanyNumber` | The unique company id that is our join key everywhere | Companies House |
| `CompanyNamw` | Human-readable name (for sanity checks only, never for joining) | Companies House |
| `news_coverage_count` | How many Guardian articles match (0 is a valid answer!) | `response.total` |
| `n_verified` | How many of those we confirmed are really about this company | our verification step |
| `articles` | The kept articles (title, url, date) | `response.results` |
| `sentiment_score` | Tone of the coverage, roughly −1 (negative) to +1 (positive) | sentiment model |
| `sentiment_label` | positive / neutral / negative | derived from the score |
| `search_date` | The date we collected the data (so results are reproducible) | pinned constant |

**2 things which i believe are important to keep the outcome trustworthy**

1. **Disambiguation is built in.** Lots of companies share names with unrelated things, and the Guardian search can return loosely-matching articles. So to *verify* every outcome against the company's town / sector before trusting it.
2. **`CompanyNumber`** Company *names* tend to clash and change; the number never does. so join on the number, never the name.


## 1. Config & constants

So Before any API calls, I set everything up in one place.

- **SEARCH_DATE** is pinned to a fixed day. The methodology chapter can then cite an exact data-collection date.
- **3 year** date window. UK SMEs dont appear much in the news, so a short window (weeks/months) would return almost nothing. A 3-year window maybe a good idea to get *any* signal (As guardian is quite vauge). We still keep each article's publication date, so we can look at recency later if we want.
- **Sections** I have hard-filtered to `business - money - technology - uk_new`. (This trades a little recall for a lot of precision)
- **Rate limits:** the free Guardian tier allows **1 call/second** and **500 calls/day**, with full article text.

In [46]:
# Imports
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

#Reproducibility: pin the data-collection date
SEARCH_DATE  = "2026-06-30"          # the day we treat as "now" for this run
WINDOW_YEARS = 3                      # how far back we look for articles

_sd = datetime.strptime(SEARCH_DATE, "%Y-%m-%d")
FROM_DATE = _sd.replace(year=_sd.year - WINDOW_YEARS).strftime("%Y-%m-%d")
TO_DATE   = SEARCH_DATE
print(f"Collecting Guardian articles from {FROM_DATE} to {TO_DATE} (a {WINDOW_YEARS}-year window)")

#Guardian endpoint & query settings
GUARDIAN_URL = "https://content.guardianapis.com/search"
SECTIONS     = "business|money|technology|uk-news"   # hard section filter ( | means OR )
SHOW_FIELDS  = "headline,trailText,standfirst,body,wordcount,byline"
SHOW_TAGS    = "keyword,tone,type"
PAGE_SIZE    = 50                    # max allowed; one page is plenty per company

#Rate limiting (free developer tier)
RATE_LIMIT_SECONDS = 1.0             # 1 call/second
DAILY_CALL_BUDGET  = 500             # 500 calls/day

#The API key is read from an environment variable, and not hardcoded.
#The earlier notebooks i pasted real keys straight into the cells, 
#which means those keys are now exposed in the git history. Set your key once in your environment (or a local `.env`) and the notebook reads it.
GUARDIAN_API_KEY = os.environ.get("GUARDIAN_API_KEY", "")
if GUARDIAN_API_KEY:
    print("Guardian API key loaded from environment.")
else:
    print("No GUARDIAN_API_KEY found. Set it before running the API cells, e.g.:")
    print('   Windows (PowerShell):  $env:GUARDIAN_API_KEY = "your-key-here"')
    print('   Or put it in a local .env file (and keep .env out of git).')

Guardian API key loaded from environment.


---
**Load the 96-company sample**

I previously build and tested on Viktor's existing sample of 96 companies, in NB08 but as I wasnt using a properly tuned query and the suggested way of use from guardian it was mostly giving false positives 

Will again try first on the same sampe for three reasons:

1. It already has a `search_name` column (the company name cleaned up for searching) and `query_templates`.
2. Its Balanced too with 24 companies each of Large - Medium - Micro - Small, evenly across the 3 sectors. 

The data lives one level up from the repo (in the outer `data/processed` folder), so I search a couple of likely locations instead of hardcoding one path. I also strip the column names, because some Companies House columns have annoying leading spaces.

In [47]:
# Find the sample (it lives in the outer data/processed folder, not inside the repo)
NB_DIR = Path.cwd()                 # .../notebooks
REPO   = NB_DIR.parent             # repo root

candidates = [
    REPO / "data" / "processed" / "nb04_stress_test_sample.csv",          # inside repo (if it ever moves here)
    REPO.parent / "data" / "processed" / "nb04_stress_test_sample.csv",   # outer Lloyds_Github/data (current home)
]

SAMPLE_PATH = next((p for p in candidates if p.exists()), None)
if SAMPLE_PATH is None:
    raise FileNotFoundError(
        "Could not find nb04_stress_test_sample.csv. Looked in:\n  "
        + "\n  ".join(str(p) for p in candidates)
    )

sample = pd.read_csv(SAMPLE_PATH)
sample.columns = sample.columns.str.strip()   # fix leading spaces in column names
print(f"Loaded {len(sample)} companies from: {SAMPLE_PATH}")

Loaded 96 companies from: C:\Users\visha\Lloyds_Github\data\processed\nb04_stress_test_sample.csv


In [48]:
# Quick look at the sample so we know exactly what we're working with
print("Rows:", len(sample))
print("\nBalance check (segment x sector):")
print(pd.crosstab(sample["segment"], sample["sector"], margins=True))

print("\nThe columns we'll lean on:")
preview_cols = ["CompanyName", "CompanyNumber", "RegAddress.PostTown", "sector", "segment", "search_name"]
preview_cols = [c for c in preview_cols if c in sample.columns]
print(sample[preview_cols].head(8).to_string(index=False))

Rows: 96

Balance check (segment x sector):
sector   Fast growth & emerging  Manufacturing  \
segment                                          
Large                         8              8   
Medium                        8              8   
Micro                         8              8   
Small                         8              8   
All                          32             32   

sector   Technology, legal & professional  All  
segment                                         
Large                                   8   24  
Medium                                  8   24  
Micro                                   8   24  
Small                                   8   24  
All                                    32   96  

The columns we'll lean on:
                                        CompanyName CompanyNumber RegAddress.PostTown                 sector segment                                                                   search_name
                                       

**What to expect:** 96 rows, a clean 8-per-cell grid (24 per size tier, 32 per sector), and a `search_name` that already has the `LTD`/`LIMITED` bits stripped off. If anything looks off here, we fix it before going near the API.

## 3. Search query

here i will turn a company into a Guardian query that finds **the right** articles.

On a side note* in the 96 sample data `search_name` column isn't just the company name, it's the name, town, and sector all stuck into one string, e.g. `whalar london fast growth & emerging`. That isfine for some uses, but it's a poor thing to search on:
- Quoting the whole string as an exact phrase matches *nothing* (no article says that exact sentence).
- Taking the first few words (check NB08) mangles longer names like e.g. *CME Technology and Support Services* becomes **cme technology and**.

So instead, I've cleaned the company name here from `CompanyName` (strip the `LTD`/`LIMITED`/etc. bits) and use that as an exact phrase. 
- The town and sector are kept **separately**
- we'll use them in the verification step to check a hit is genuine, rather than forcing into the search.

Q) **Why keep town/sector out of the search itself?** 
- If I demanded the article also contain the town, I wuold throw away real articles that just don't happen to mention it (a story about a company often won't name its town).
- And for London companies the town is a bit of weak filter as "London" appears in half the paper.
- Better to search precisely on the name, then verify afterwards. 

The query itself uses:
- `q = "<clean name>"` : exact phrase, so *Acme Joinery* doesn't match *acme* + *joinery* scattered about.
- `query-fields=body` : the name must appear in the **article body**, and not just a loose field match.

> **Heads-up on special characters.** Some company names contain `&`, `(`, `)` or `|`, which the Guardian `q` parameter reads as *operators* - so a name like `J.A. Harrison & Company (Manchester)` returns a `400 Bad Request`. The `to_query_phrase()` function below makes the name safe: it drops bracketed bits, turns `&` into 'and', and keeps only letters, numbers and spaces.

In [49]:
import re

# Legal-form endings to strip so "ACME JOINERY LIMITED" -> "Acme Joinery"
LEGAL_SUFFIXES = [
    " LIMITED", " LTD", " PLC", " LLP", " LP", " CIC",
    " INC", " CORP", " CORPORATION", " GROUP", " HOLDINGS", " COMPANY", " CO",
]

def clean_company_name(raw_name: str) -> str:
    """Strip legal suffixes and trailing punctuation to get a readable name."""
    name = str(raw_name).upper().strip().rstrip(" ,.")
    changed = True
    while changed:                      # loop handles stacked endings like ", LTD"
        changed = False
        for suffix in LEGAL_SUFFIXES:
            if name.endswith(suffix):
                name = name[:-len(suffix)].strip().rstrip(" ,.")
                changed = True
    return name.title()                 # Guardian search is case-insensitive; .title() just reads nicely

def to_query_phrase(clean_name: str) -> str:
    """Make a name SAFE for the Guardian `q` parameter.

    The q syntax treats & ( ) | and other punctuation as operators, so a raw name like
    'J.A. Harrison & Company (Manchester)' causes a 400 Bad Request. We therefore:
      - drop bracketed qualifiers like '(Manchester)',
      - turn '&' into 'and',
      - keep only letters, numbers and spaces.
    """
    name = re.sub(r"\(.*?\)", " ", clean_name)     # remove "(Manchester)" style bits
    name = name.replace("&", " and ")
    name = re.sub(r"[^A-Za-z0-9 ]+", " ", name)     # strip any remaining punctuation
    name = re.sub(r"\s+", " ", name).strip()
    return name

# Sector -> a few words we'd expect in a genuine article (used later, in verification)
SECTOR_KEYWORDS = {
    "Manufacturing":                    ["manufacturing", "factory", "production", "engineering"],
    "Technology, legal & professional": ["technology", "software", "legal", "consultancy", "services"],
    "Fast growth & emerging":           ["startup", "funding", "investment", "scale-up", "growth"],
}

def build_query(clean_name: str) -> str:
    """The Guardian q value: the cleaned, query-safe name as an exact phrase.
    Returns '' if the name is too short/garbled to search safely."""
    phrase = to_query_phrase(clean_name)
    return f'"{phrase}"' if len(phrase) >= 3 else ""

# Build the readable name and the actual search phrase
sample["clean_name"]    = sample["CompanyName"].apply(clean_company_name)
sample["search_phrase"] = sample["clean_name"].apply(to_query_phrase)

# Show some normal names, then specifically the tricky ones with & or brackets,
# so we can confirm the sanitising works on the names that caused the 400.
print("Normal names:")
print(sample[["CompanyName", "search_phrase"]].head(5).to_string(index=False))

tricky = sample[sample["CompanyName"].str.contains(r"[&()]", regex=True, na=False)]
if len(tricky):
    print("\nTricky names (had & or brackets) - now made query-safe:")
    print(tricky[["CompanyName", "search_phrase"]].head(8).to_string(index=False))

Normal names:
                                        CompanyName                                  search_phrase
                                         WHALAR LTD                                         Whalar
                                            NCB RIP                                        Ncb Rip
EXODUSPOINT CAPITAL MANAGEMENT UK TECHNOLOGIES, LTD Exoduspoint Capital Management Uk Technologies
                               AKSIA EUROPE LIMITED                                   Aksia Europe
        CME TECHNOLOGY AND SUPPORT SERVICES LIMITED            Cme Technology And Support Services

Tricky names (had & or brackets) - now made query-safe:
                                 CompanyName            search_phrase
J.A. HARRISON & COMPANY (MANCHESTER) LIMITED J A Harrison and Company
              SHAW TIMBER (HOLDINGS) LIMITED              Shaw Timber
                ADVANIA UK (SERVIUM) LIMITED               Advania Uk
       RADIO COMPUTING SERVICES (UK) LIMITED Radio Comp

---
## 4.fetch function (rate-limited + cached)

This wraps the actual API call. (Two imp things)

- **Caching.** The first time we look up a company, we save the raw JSON response to disk (`data/raw/search_cache/<CompanyNumber>_guardian.json`). Every run after that reads from the file instead of calling the API. This means: (1) we don't burn our 500/day budget re-running cells, and (2) results are reproducible.
  
- **Rate limiting.** We sleep 1 second after each *real* API call (the free tier allows 1/second). Cached lookups don't sleep, so re-runs are quick.



In [50]:
# Cache lives next to the data (.../data/raw/search_cache), wherever the data root actually is
CACHE_DIR = SAMPLE_PATH.parent.parent / "raw" / "search_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Cache folder:", CACHE_DIR)

EMPTY_RESPONSE = {"response": {"total": 0, "results": []}}

def guardian_search(company_number: str, clean_name: str, api_key: str, use_cache: bool = True) -> tuple:
    """Look up one company on the Guardian.

    Returns (raw_response_dict, source). source is one of:
      'cache' - read from disk          'api'   - freshly fetched and cached
      'skip'  - name unsearchable        'error' - the API returned a non-200 (not cached, so a re-run retries)
    It never raises on a bad single company, so one odd name can't halt the whole 96-scan.
    """
    cache_file = CACHE_DIR / f"{company_number}_guardian.json"

    # 1) Use the cached copy if we already have one
    if use_cache and cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8")), "cache"

    # 2) Skip names that sanitised down to nothing searchable
    query = build_query(clean_name)
    if not query:
        return EMPTY_RESPONSE, "skip"

    # 3) Call the API
    params = {
        "q":            query,
        "query-fields": "body",          # name must appear in the article body
        "section":      SECTIONS,        # business|money|technology|uk-news
        "from-date":    FROM_DATE,
        "to-date":      TO_DATE,
        "lang":         "en",
        "order-by":     "relevance",
        "page-size":    PAGE_SIZE,
        "show-fields":  SHOW_FIELDS,
        "show-tags":    SHOW_TAGS,
        "api-key":      api_key,
    }
    resp = requests.get(GUARDIAN_URL, params=params, timeout=30)

    if resp.status_code != 200:
        print(f"   [warning] {resp.status_code} for {clean_name!r} -> skipped (not cached)")
        return EMPTY_RESPONSE, "error"   # don't cache errors

    data = resp.json()
    cache_file.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    time.sleep(RATE_LIMIT_SECONDS)
    return data, "api"

def summarise_response(data: dict) -> tuple:
    """Pull the headline numbers out of a raw response: (total_matches, returned_now, first_few_titles)."""
    r = data.get("response", {})
    total   = r.get("total", 0)
    results = r.get("results", [])
    titles  = [a.get("webTitle", "") for a in results[:3]]
    return total, len(results), titles

Cache folder: C:\Users\visha\Lloyds_Github\data\raw\search_cache


## 4b. Doing a test on a few companies

I'll run it on the first 3 companies just to see it working end-to-end. This costs **3 API calls** (out of 500/day) the first time; after that it reads from the cache and is free.

something i lookout for is two things: the **coverage count** (`total`), and whether the returned titles actually look like they're about our company.

In [51]:
def diagnostic_count(query, query_fields=None, section=None):
    """Lightweight one-off count for testing the query mechanics (not cached, page-size=1)."""
    params = {"q": query, "from-date": FROM_DATE, "to-date": TO_DATE,
              "lang": "en", "page-size": 1, "api-key": GUARDIAN_API_KEY}
    if query_fields:
        params["query-fields"] = query_fields
    if section:
        params["section"] = section
    r = requests.get(GUARDIAN_URL, params=params, timeout=30)
    r.raise_for_status()
    time.sleep(RATE_LIMIT_SECONDS)
    return r.json().get("response", {}).get("total", 0)

assert GUARDIAN_API_KEY, "Set GUARDIAN_API_KEY first (see Section 1)."

q = '"Tesco"'
print("Known-big-company test (Tesco), watching each filter narrow the results:")
print(f"  name only ............................. {diagnostic_count(q):>6} articles")
print(f"  + section filter ...................... {diagnostic_count(q, section=SECTIONS):>6} articles")
print(f"  + section + must appear in body ....... {diagnostic_count(q, query_fields='body', section=SECTIONS):>6} articles")

Known-big-company test (Tesco), watching each filter narrow the results:
  name only .............................   1209 articles
  + section filter ......................    571 articles
  + section + must appear in body .......    585 articles


**So to explain in plain words**
- If Tesco returns lots of articles (hundreds/thousands), the API call, key, date window and filters all work - so the zeros for our small companies are **real**, not a bug.
- You should see the count get smaller as each filter is added. That's expected and healthy - `query-fields=body` in particular drops articles that only mention "Tesco" in passing metadata.
- If even Tesco returns 0, something's wrong (key, date format, or a filter typo) and we fix that before going further.

## 5. Now below check coverage all 96 companies

Now i will run the real fetch on the whole sample to answer: *how many of these 96 companies have any Guardian coverage at all, and does it depend on company size?*

This is ~96 API calls the first time (pls check with the limits when you run it twice) We're only collecting the **coverage count** here and finding articles is one job, deciding which are genuine is the next (verification).

In [52]:
assert GUARDIAN_API_KEY, "Set GUARDIAN_API_KEY first."

counts, sources = [], []
for i, (_, row) in enumerate(sample.iterrows(), start=1):
    data, source = guardian_search(row["CompanyNumber"], row["clean_name"], GUARDIAN_API_KEY)
    total, _, _ = summarise_response(data)
    counts.append(total)
    sources.append(source)
    if i % 20 == 0:
        print(f"  ...{i}/{len(sample)} done")

sample["coverage_count"] = counts
print(f"\nDone. api={sources.count('api')}, cache={sources.count('cache')}, "
      f"skip={sources.count('skip')}, error={sources.count('error')}")
if sources.count('error'):
    print("Some companies errored (left uncached) - just re-run this cell to retry them.")

  ...20/96 done
  ...40/96 done
  ...60/96 done
  ...80/96 done

Done. api=0, cache=96, skip=0, error=0


In [53]:
# How much coverage did we actually find?
n_cov = int((sample["coverage_count"] > 0).sum())
print(f"Companies with at least 1 Guardian article: {n_cov} / {len(sample)} ({n_cov/len(sample):.1%})")
print(f"Total articles across all companies: {int(sample['coverage_count'].sum())}")

print("\nCoverage by size tier:")
tier = sample.groupby("segment").agg(
    companies     = ("CompanyNumber", "count"),
    with_coverage = ("coverage_count", lambda s: int((s > 0).sum())),
    total_articles= ("coverage_count", "sum"),
)
tier["coverage_rate"] = (tier["with_coverage"] / tier["companies"] * 100).round(1)
print(tier.to_string())

print("\nThe companies that DO have coverage (these are what verification will work on):")
covered = sample[sample["coverage_count"] > 0].sort_values("coverage_count", ascending=False)
print(covered[["clean_name", "segment", "sector", "RegAddress.PostTown", "coverage_count"]].head(20).to_string(index=False))

Companies with at least 1 Guardian article: 5 / 96 (5.2%)
Total articles across all companies: 83

Coverage by size tier:
         companies  with_coverage  total_articles  coverage_rate
segment                                                         
Large           24              2               4            8.3
Medium          24              1              62            4.2
Micro           24              1              16            4.2
Small           24              1               1            4.2

The companies that DO have coverage (these are what verification will work on):
         clean_name segment                           sector RegAddress.PostTown  coverage_count
           Baroness  Medium Technology, legal & professional                 NaN              62
            Coaster   Micro           Fast growth & emerging              LONDON              16
                Ncc   Large Technology, legal & professional          MANCHESTER               3
           Brs Golf

## 6. Verification / disambiguation



The main idea here is that an article is *verified* if the company name appears as a phrase **and** there's at least one corroborating signal:
- *T* : the company's town appears in the article,
- *S* : a sector keyword appears (from `SECTOR_KEYWORDS`),
- *C* : general business-context words appear (company, firm, CEO, turnover, ...).

**Single-word names are treated more strictly.** A name like "Baroness" or "Coaster" is one common word, so business-context alone isn't enough, so we require the **town** to be present too.

In [54]:
import re
from html import unescape

# Words that suggest an article is about a *business* (separates a company from a same-named person/thing)
BUSINESS_CONTEXT = [
    "company", "firm", "business", "ltd", "limited", "plc", "group", "holdings",
    "ceo", "chief executive", "founder", "director", "turnover", "revenue", "profit",
    "acquisition", "merger", "contract", "customers", "employees", "staff", "headquarters",
]

# Fix 3: common-word company names that can't be matched by name reliably (extend when you spot one)
UNSEARCHABLE_NAMES = {"coaster", "baroness"}

def _article_text(article: dict) -> str:
    """Whole article (title + all fields), lowercase, tag-free - used for corroboration signals."""
    f = article.get("fields", {})
    blob = " ".join([article.get("webTitle", ""), f.get("headline", ""), f.get("standfirst", ""),
                     f.get("trailText", ""), f.get("body", "")])
    return unescape(re.sub(r"<[^>]+>", " ", blob)).lower()

def _summary_text(article: dict) -> str:
    """Only the summary fields (title/headline/standfirst/trail) - where an article names what it is ABOUT."""
    f = article.get("fields", {})
    blob = " ".join([article.get("webTitle", ""), f.get("headline", ""),
                     f.get("standfirst", ""), f.get("trailText", "")])
    return unescape(re.sub(r"<[^>]+>", " ", blob)).lower()

def _mentions(text: str, phrase: str) -> bool:
    """Fix 1: True only if phrase stands alone - not inside another word or hyphenated.
    So 'coaster' no longer matches 'roller-coaster', which was the main false-positive source."""
    return re.search(r"(?<![\w-])" + re.escape(phrase) + r"(?![\w-])", text) is not None

def verify_article(article: dict, clean_name: str, town, sector: str) -> dict:
    name = clean_name.lower().strip()
    summary, full = _summary_text(article), _article_text(article)

    unsearchable = name in UNSEARCHABLE_NAMES
    # Fix 2: the name must appear (standalone) in the SUMMARY, not just buried in the body.
    # An article genuinely about a company names it up front; an oil-market piece that merely
    # contains the word in passing will not have it in the headline/standfirst/trail.
    phrase     = _mentions(summary, name)
    town_ok    = isinstance(town, str) and len(town) > 1 and _mentions(full, town.lower())
    sector_ok  = any(_mentions(full, k) for k in SECTOR_KEYWORDS.get(sector, []))
    context_ok = any(c in full for c in BUSINESS_CONTEXT)
    single     = len(re.findall(r"[a-z0-9]+", name)) <= 1

    verified = (not unsearchable) and phrase and (town_ok or sector_ok or context_ok)

    return {"phrase": phrase, "town": town_ok, "sector": sector_ok, "context": context_ok,
            "single_word": single, "unsearchable": unsearchable, "verified": verified}

def verify_company(row, api_key) -> list:
    """Run verification over all of a company's cached articles. Returns a list of per-article dicts."""
    data, _ = guardian_search(row["CompanyNumber"], row["clean_name"], api_key)   # cache hit -> free
    out = []
    for a in data.get("response", {}).get("results", []):
        v = verify_article(a, row["clean_name"], row.get("RegAddress.PostTown"), row["sector"])
        out.append({"date": a.get("webPublicationDate", "")[:10],
                    "title": a.get("webTitle", ""), "url": a.get("webUrl", ""), **v})
    return out

print("Verification updated: standalone match in summary fields + common-word block. Flags: P=phrase(summary), T=town, S=sector, C=context.")

Verification updated: standalone match in summary fields + common-word block. Flags: P=phrase(summary), T=town, S=sector, C=context.


In [55]:
# Apply verification to every company that had coverage, and print the reasoning per article
assert GUARDIAN_API_KEY, "Set GUARDIAN_API_KEY first."

covered = sample[sample["coverage_count"] > 0].sort_values("coverage_count", ascending=False)
verify_records = {}

for _, row in covered.iterrows():
    recs = verify_company(row, GUARDIAN_API_KEY)
    verify_records[row["CompanyNumber"]] = recs
    n_keep = sum(r["verified"] for r in recs)
    town = row.get("RegAddress.PostTown")
    print(f"\n=== {row['clean_name']}  ({row['segment']}, town={town})  "
          f"{len(recs)} articles -> {n_keep} verified ===")
    for r in recs[:12]:
        flags = ("P" if r["phrase"] else "-") + ("T" if r["town"] else "-") + \
                ("S" if r["sector"] else "-") + ("C" if r["context"] else "-")
        mark = "KEEP" if r["verified"] else "drop"
        print(f"  [{mark}] {flags}  {r['date']}  {r['title'][:70]}")
    if len(recs) > 12:
        print(f"   ... and {len(recs) - 12} more")


=== Baroness  (Medium, town=nan)  50 articles -> 0 verified ===
  [drop] --SC  2025-10-03  Michelle Mone says she has ‘no wish’ to remain a Conservative peer
  [drop] --SC  2025-12-08  Grooming gangs inquiry ‘must consider ethnicity and religion’, Badenoc
  [drop] ----  2026-01-21  Lords put pressure on Starmer with vote to ban social media for under-
  [drop] --SC  2025-06-16  Police to collect ethnicity data for all cases of child sexual abuse
  [drop] ----  2025-11-28  Met police to face ‘Casey 2’ inquiry amid recent scandals
  [drop] ---C  2025-12-25  ‘Lost decade’ of progress after UK introduced shared parental leave, s
  [drop] ---C  2025-11-07  Met police’s culture makes racial harm ‘inevitable’, internal review f
  [drop] ---C  2025-10-23  Grooming gangs inquiry divided over the question of widening its focus
  [drop] --S-  2025-10-20  Starmer’s grooming gang inquiry left in turmoil after two survivors qu
  [drop] ---C  2025-11-25  Covid inquiry lays bare unforgivable failures

In [56]:
# Roll the verified counts back onto the sample and compare raw vs verified
sample["n_verified"] = sample["CompanyNumber"].map(
    lambda cn: sum(r["verified"] for r in verify_records.get(cn, []))
)

n_raw = int((sample["coverage_count"] > 0).sum())
n_ver = int((sample["n_verified"] > 0).sum())
print(f"Companies with RAW coverage:      {n_raw} / {len(sample)}")
print(f"Companies with VERIFIED coverage: {n_ver} / {len(sample)}")
print(f"Articles:  raw = {int(sample['coverage_count'].sum())}  ->  verified = {int(sample['n_verified'].sum())}")

print("\nWhat survived verification:")
survivors = sample[sample["n_verified"] > 0]
cols = ["clean_name", "segment", "sector", "RegAddress.PostTown", "coverage_count", "n_verified"]
print(survivors[cols].to_string(index=False) if len(survivors) else "  (nothing passed - all raw hits were collisions)")

Companies with RAW coverage:      5 / 96
Companies with VERIFIED coverage: 0 / 96
Articles:  raw = 83  ->  verified = 0

What survived verification:
  (nothing passed - all raw hits were collisions)



- Each article shows four flags - **P**hrase, **T**own, **S**ector, **C**ontext - and a `KEEP`/`drop` decision.
- I expect **Baroness** and **Coaster** to be almost entirely `drop` (they're the word-collisions), and **NCC** to keep its genuine articles (Manchester cyber firm).

## 7. Sentiment (FinBERT)

Now we score the *verified* articles for tone. We use **FinBERT** (`ProsusAI/finbert`) rather than a general scorer like VADER, because FinBERT is trained on **financial / business text** - so it reads sentences like "profits fell sharply" or "secured new funding" the way a banker would, not the way a tweet would. That fits a project about company risk and opportunity, and it matches the NB04 spec.

A few practical notes:
- **One-off install:** `pip install transformers torch`. The first run also downloads the model (~400 MB), then it's cached locally. Our data is tiny, so it runs fine on a normal laptop CPU.
- **What text we score:** the short, clean fields - `headline` + `standfirst` + `trailText`. These summarise the article well and stay within FinBERT's length limit (long bodies are truncated automatically).
- **Turning the label into a number:** FinBERT returns `positive` / `negative` / `neutral` with a confidence. We map that to a single signed score: positive -> `+confidence`, negative -> `-confidence`, neutral -> `0`. A company's score is the **average** across its verified articles, in roughly -1..+1.

In [57]:
# FinBERT is loaded once, lazily, so importing this cell is cheap and we only pay the cost when we score.
_finbert = None

def get_finbert():
    global _finbert
    if _finbert is None:
        try:
            from transformers import pipeline
        except ImportError:
            raise ImportError("FinBERT needs transformers + torch. Install once with:  pip install transformers torch")
        print("Loading FinBERT (first run downloads ~400 MB, then it's cached)...")
        _finbert = pipeline("text-classification", model="ProsusAI/finbert", truncation=True)
        print("FinBERT ready.")
    return _finbert

def sentiment_text(article: dict) -> str:
    """Short, clean text for FinBERT: headline + standfirst + trailText (HTML stripped)."""
    f = article.get("fields", {})
    blob = " ".join([f.get("headline", "") or article.get("webTitle", ""),
                     f.get("standfirst", ""), f.get("trailText", "")])
    return unescape(re.sub(r"<[^>]+>", " ", blob)).strip()

def score_text(text: str) -> tuple:
    """Return (signed_score in -1..+1, label) for one piece of text."""
    if not text or len(text.strip()) < 5:
        return 0.0, "neutral"
    res = get_finbert()(text[:2000])[0]          # char cap + the pipeline's own token truncation
    label = res["label"].lower()
    conf  = float(res["score"])
    signed = conf if label == "positive" else (-conf if label == "negative" else 0.0)
    return round(signed, 3), label

def verified_articles(row, api_key) -> list:
    """The full article dicts (from cache) that passed verification, for scoring."""
    data, _ = guardian_search(row["CompanyNumber"], row["clean_name"], api_key)
    return [a for a in data.get("response", {}).get("results", [])
            if verify_article(a, row["clean_name"], row.get("RegAddress.PostTown"), row["sector"])["verified"]]

def company_sentiment(row, api_key) -> tuple:
    """Average FinBERT score across a company's verified articles -> (score, label, n_scored)."""
    arts = verified_articles(row, api_key)
    if not arts:
        return None, "no_coverage", 0
    scores = [score_text(sentiment_text(a))[0] for a in arts]
    mean = round(sum(scores) / len(scores), 3)
    label = "positive" if mean > 0.05 else ("negative" if mean < -0.05 else "neutral")
    return mean, label, len(arts)

print("Sentiment functions defined.")

Sentiment functions defined.


In [58]:
# Score only the companies that have verified coverage (a handful) - everything else is left as no_coverage.
assert GUARDIAN_API_KEY, "Set GUARDIAN_API_KEY first."

sent_score, sent_label = {}, {}
to_score = sample[sample["n_verified"] > 0]
print(f"Scoring {len(to_score)} companies with verified coverage...\n")

for _, row in to_score.iterrows():
    score, label, n = company_sentiment(row, GUARDIAN_API_KEY)
    sent_score[row["CompanyNumber"]] = score
    sent_label[row["CompanyNumber"]] = label
    print(f"  {row['clean_name']:<22} {n} article(s) -> score={score:+.3f}  ({label})")

Scoring 0 companies with verified coverage...



**Reading this:** each company's score is the average tone of its verified articles, from about -1 (very negative) to +1 (very positive), with 0 = neutral/mixed. With so few articles these are illustrative rather than statistically strong - which is exactly the honest framing for the write-up: the *method* is the deliverable, and the thin coverage is the finding.

## 8. Assemble the per-company rows and export

Finally we pull everything into **one tidy row per company** - the output the rest of the pipeline (and the structured side) can join onto. Every company in the sample gets a row, including the many with no coverage (their `news_coverage_count` is 0 and sentiment is blank - remember, zero coverage is a valid result, not a gap).

We save it to `data/processed/nb09_guardian_signals.csv` - a **new** file, so we don't touch NB08's output. `CompanyNumber` stays the join key throughout.

In [59]:
# Build one row per company
def kept_titles(cn):
    kept = [r for r in verify_records.get(cn, []) if r["verified"]]
    return " | ".join(r["title"] for r in kept[:3])

signals = pd.DataFrame({
    "CompanyNumber":       sample["CompanyNumber"],
    "CompanyName":         sample["CompanyName"],
    "clean_name":          sample["clean_name"],
    "segment":             sample["segment"],
    "sector":              sample["sector"],
    "town":                sample["RegAddress.PostTown"],
    "news_coverage_count": sample["coverage_count"],          # raw Guardian matches
    "n_verified":          sample["n_verified"],              # genuinely about the company
    "sentiment_score":     sample["CompanyNumber"].map(sent_score),
    "sentiment_label":     sample["CompanyNumber"].map(sent_label).fillna("no_coverage"),
    "verified_titles":     sample["CompanyNumber"].map(kept_titles),
    "search_date":         SEARCH_DATE,
})

print(f"Built signals for {len(signals)} companies.")
print("\nCompanies with verified coverage:")
print(signals[signals["n_verified"] > 0].to_string(index=False))

Built signals for 96 companies.

Companies with verified coverage:
Empty DataFrame
Columns: [CompanyNumber, CompanyName, clean_name, segment, sector, town, news_coverage_count, n_verified, sentiment_score, sentiment_label, verified_titles, search_date]
Index: []


In [60]:

# Quick recap of what the tool produced overall
print(f"  companies with raw coverage      : {int((signals['news_coverage_count']>0).sum())}")
print(f"  companies with verified coverage : {int((signals['n_verified']>0).sum())}")
print(f"  total raw articles               : {int(signals['news_coverage_count'].sum())}")
print(f"  total verified articles          : {int(signals['n_verified'].sum())}")

  companies with raw coverage      : 5
  companies with verified coverage : 0
  total raw articles               : 83
  total verified articles          : 0


## The Guardian tool is complete

We now have a reusable unit: **company in -> articles found, coverage count, verification, sentiment, one tidy row out**, saved to `nb09_guardian_signals.csv`. It runs end-to-end on the 96-company sample, is cached and reproducible, and uses `CompanyNumber` throughout.

**The honest takeaway:** even a quality UK source like the Guardian covers only a sliver of these companies, and most of the raw hits were name-collisions removed by verification. That sparsity is a genuine finding - and it's the reason the plan always called for a **second source**.

**Next:** point the *same* structure at another API for breadth - **GDELT** (free, huge, includes tone; check whether the earlier IP ban has lifted) or **GNews** as the easy fallback. Only the fetch function changes; verification, sentiment and assembly stay the same.

## What's next (after you review this part)

Built so far: framing + config + sample loaded, the query design (clean name + exact phrase in body), a cached/rate-limited fetch, a sanity check, and a coverage scan over all 96.

Next layers (one at a time, after your review):

1. **Verification / disambiguation** - for each *covered* company, open its articles and keep only the ones genuinely about that company (name in the text + a corroborating signal: town, sector keyword, or business context). Produces `n_verified` and the trustworthy article list.
2. **Sentiment** - score the verified text (we'll decide FinBERT vs VADER deliberately and write down why).
3. **Assemble one row per company** and finalise across all 96.
4. **Sanity-check & compare** to NB08, and look at coverage by size tier.

Pausing here. Tell me the scan results (how many covered, and roughly which) and we'll build verification around the real articles.